In [1]:
import numpy as np
import pandas as pd

In [2]:
## Phase 5: EWMA scaled historical simulation Var and ES

In [3]:
# Step 1: Loading phase 2 output
portfolio_dollar_value = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\portfolio_dollar_value.csv', index_col = 0, parse_dates = True).iloc[:,0]
portfolio_pnl = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\portfolio_pnl.csv', index_col = 0, parse_dates = True).iloc[:,0]


In [4]:
# Simple returns for actual dollar scaled Var projections
port_simple_returns = portfolio_dollar_value.pct_change().dropna()

In [5]:
# Log return for volatility estimations. 
port_log_returns = np.log(portfolio_dollar_value / portfolio_dollar_value.shift(1)).dropna()

In [6]:
# Step 2: Full EWMA volatility model 

In [7]:
LAMBDA = 0.94 # Standard daily decay factor

In [8]:
# F_{t-1} measurability: Shifting squared log return by 1 before feeding to .ewm() 
# so that the volatility forecast of day t uses data strictly through day t-1
sq_log_returns_shifted = (port_log_returns**2).shift(1).dropna()
ewma_variance = sq_log_returns_shifted.ewm(alpha = (1 - LAMBDA), adjust = False). mean()
ewma_vol = np.sqrt(ewma_variance)


In [9]:
# Aligning simple return and EWMA volatility to a common date index
common_idx = port_simple_returns.index.intersection(ewma_vol.index)
assert len(common_idx) > 0
port_simple_returns_aligned = port_simple_returns.loc[common_idx]
ewma_vol_aligned = ewma_vol.loc[common_idx]
print(f'Total available trading days for EWMA engine is {len(port_simple_returns)}')


Total available trading days for EWMA engine is 1506


In [10]:
# Step 3: Parameters & Core Risk Function (reused from Phase 4)

In [11]:
windows = [250, 500, 750]
confidence_levels = [0.95, 0.99, 0.995]
MAX_SCALING_RATIO = 3.0 
def calculate_historical_var_es(returns_window, conf_level):
    """Unmodified from Phase 4 - same engine, different input returns."""
    percentile = (1 - conf_level) * 100
    var_pct = -np.percentile(returns_window, percentile, method='lower')
    tail_returns = returns_window[returns_window <= -var_pct]
    es_pct = -tail_returns.mean() if len(tail_returns) > 0 else var_pct
    return var_pct, es_pct

In [12]:
# Step 4: Initializing the Rolling Engine 

In [13]:
ewma_metrics = pd.DataFrame(index=port_simple_returns_aligned.index)
ewma_metrics['Portfolio_Dollar_Value'] = portfolio_dollar_value.loc[port_simple_returns_aligned.index]
ewma_metrics['Daily_PnL'] = portfolio_pnl.loc[port_simple_returns_aligned.index]
ewma_metrics['Portfolio_Dollar_Value_Prior'] = ewma_metrics['Portfolio_Dollar_Value'].shift(1)
ewma_metrics['EWMA_Vol_Daily'] = ewma_vol_aligned
 
ret_array = port_simple_returns_aligned.values
vol_array = ewma_vol_aligned.values

In [14]:
# Step 5: Executing Rolling Volatility-Scaled Calculations

In [15]:
for w in windows:
    for cl in confidence_levels:
        print(f"Calculating EWMA-Scaled {w}-day window at {cl * 100}% confidence...")
 
        var_list = np.full(len(ret_array), np.nan)
        es_list = np.full(len(ret_array), np.nan)
 
        for i in range(w, len(ret_array)):
            window_returns = ret_array[i - w:i]
            window_vols = vol_array[i - w:i]
            current_vol_forecast = vol_array[i]
 
            scaling_ratios = current_vol_forecast / window_vols
            scaling_ratios = np.clip(scaling_ratios, a_min=None, a_max=MAX_SCALING_RATIO)
            scaled_window_returns = window_returns * scaling_ratios
 
            v, e = calculate_historical_var_es(scaled_window_returns, cl)
            var_list[i] = v
            es_list[i] = e
 
        col_base = f"{w}d_{cl * 100}%"
        ewma_metrics[f"EWMA_VaR_pct_{col_base}"] = var_list
        ewma_metrics[f"EWMA_ES_pct_{col_base}"] = es_list
 
        ewma_metrics[f"EWMA_VaR_1d_$_" + col_base] = (
            ewma_metrics[f"EWMA_VaR_pct_{col_base}"] * ewma_metrics['Portfolio_Dollar_Value_Prior']
        )
        ewma_metrics[f"EWMA_ES_1d_$_" + col_base] = (
            ewma_metrics[f"EWMA_ES_pct_{col_base}"] * ewma_metrics['Portfolio_Dollar_Value_Prior']
        )
        ewma_metrics[f"EWMA_VaR_10d_$_" + col_base] = ewma_metrics[f"EWMA_VaR_1d_$_" + col_base] * np.sqrt(10)
        ewma_metrics[f"EWMA_ES_10d_$_" + col_base] = ewma_metrics[f"EWMA_ES_1d_$_" + col_base] * np.sqrt(10)

Calculating EWMA-Scaled 250-day window at 95.0% confidence...
Calculating EWMA-Scaled 250-day window at 99.0% confidence...
Calculating EWMA-Scaled 250-day window at 99.5% confidence...
Calculating EWMA-Scaled 500-day window at 95.0% confidence...
Calculating EWMA-Scaled 500-day window at 99.0% confidence...
Calculating EWMA-Scaled 500-day window at 99.5% confidence...
Calculating EWMA-Scaled 750-day window at 95.0% confidence...
Calculating EWMA-Scaled 750-day window at 99.0% confidence...
Calculating EWMA-Scaled 750-day window at 99.5% confidence...


In [16]:
# Step 6: Finalizing the Out-of-Sample Dataset 

In [17]:
out_of_sample_ewma = ewma_metrics.dropna(subset=['EWMA_VaR_1d_$_750d_99.0%'])
 
stray_nan_count = out_of_sample_ewma.isna().sum().sum()
if stray_nan_count > 0:
    print(f"WARNING: {stray_nan_count} unexpected NaN value(s) remain outside the checked column.")
else:
    print("No stray NaN values found elsewhere in the out-of-sample dataset.")
 
print(f"\nPhase 5 Complete. Out-of-sample backtesting days available: {len(out_of_sample_ewma)}")

No stray NaN values found elsewhere in the out-of-sample dataset.

Phase 5 Complete. Out-of-sample backtesting days available: 755


In [18]:
# Step 7: Validations (checkeing across ALL out-of-sample days)

In [19]:
var_1d = out_of_sample_ewma['EWMA_VaR_1d_$_750d_99.0%']
var_10d = out_of_sample_ewma['EWMA_VaR_10d_$_750d_99.0%']
es_1d = out_of_sample_ewma['EWMA_ES_1d_$_750d_99.0%']
 
assert (var_1d > 0).all(), "Validation Failed: EWMA VaR must be positive on every day."
assert (var_10d > var_1d).all(), "Validation Failed: 10-day EWMA VaR must exceed 1-day on every day."
assert (es_1d >= var_1d).all(), "Validation Failed: EWMA ES must be >= EWMA VaR on every day."
 
var_95 = out_of_sample_ewma['EWMA_VaR_1d_$_750d_95.0%']
var_99 = out_of_sample_ewma['EWMA_VaR_1d_$_750d_99.0%']
var_995 = out_of_sample_ewma['EWMA_VaR_1d_$_750d_99.5%']
 
assert (var_99 >= var_95).all(), "Validation Failed: 99% EWMA VaR must be >= 95% on every day."
assert (var_995 >= var_99).all(), "Validation Failed: 99.5% EWMA VaR must be >= 99% on every day."
 
print("All mathematical constraints validated across the full out-of-sample period.")

All mathematical constraints validated across the full out-of-sample period.


In [20]:
# Step 8: Save results for dashboarding and Phase 8

In [21]:
out_of_sample_ewma.to_csv('ewma_scaled_var_es.csv')
print("Saved: ewma_scaled_var_es.csv")

Saved: ewma_scaled_var_es.csv
